# 11 - Privacy Utility Trade-off

This notebook compares utility and linkability using exactly the same segment-level dataset.


## Same Data Principle

This stage uses the same ECG segments for both tasks.

What stays the same:
- same segment-level rows
- same `patient_id`
- same `segment_id`
- same segmentation configuration (`window = 2.0s`, `step = 1.0s`)

What changes between tasks:
- utility target: `utility_label`
- linkability target: pair label (`same patient` vs `different patient`)


In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from config import FEATURE_SETS_DIR, FINAL_SEGMENT_FEATURES_DIR
from modeling import run_linkability_baselines, run_utility_baselines


In [3]:
MAX_CHUNKS = None  # set an integer here for quick tests
FULL_FEATURE_MODELS = ["LogisticRegression"]
SUBSET_FEATURE_MODELS = ["LogisticRegression"]


## Load Full Segment Dataset


In [4]:
manifest_path = FINAL_SEGMENT_FEATURES_DIR / "manifest.json"
errors_path = FINAL_SEGMENT_FEATURES_DIR / "errors.csv"

manifest = pd.read_json(manifest_path, typ="series")
chunk_files = [FINAL_SEGMENT_FEATURES_DIR / chunk["chunk_file"] for chunk in manifest["chunks"]]
if MAX_CHUNKS is not None:
    chunk_files = chunk_files[:MAX_CHUNKS]

full_features_df = pd.concat(
    [pd.read_csv(chunk_file, compression="gzip", low_memory=False) for chunk_file in chunk_files],
    ignore_index=True,
)
errors_df = pd.read_csv(errors_path) if errors_path.exists() and errors_path.stat().st_size > 0 else pd.DataFrame()

print("Window (s):", manifest.get("window_sec"))
print("Step (s):", manifest.get("step_sec"))
print("Full feature dataset:", full_features_df.shape)
print("Stored preprocessing errors:", len(errors_df))


Window (s): 2.0
Step (s): 1.0
Full feature dataset: (406359, 215)
Stored preprocessing errors: 1


## Load Utility Top-150 Dataset


In [5]:
top150_path = FEATURE_SETS_DIR / "utility_top150_dataset.csv.gz"
selected_features_path = FEATURE_SETS_DIR / "utility_top150_features.csv"

top150_features_df = pd.read_csv(top150_path, compression="gzip", low_memory=False)
selected_features_df = pd.read_csv(selected_features_path)

print("Top-150 dataset:", top150_features_df.shape)
print("Selected utility features:", len(selected_features_df))
selected_features_df.head()


Top-150 dataset: (406359, 157)
Selected utility features: 150


,feature
0,global_max_energy
1,lead_V5_std
2,lead_V1_rms
3,lead_V6_std
4,lead_V5_rms


## Verify Same Segments


In [6]:
full_keys = full_features_df[["patient_id", "segment_id", "segment_ref", "start_sample", "end_sample"]].copy()
top150_keys = top150_features_df[["patient_id", "segment_id", "segment_ref", "start_sample", "end_sample"]].copy()

same_rows = full_keys.equals(top150_keys)
print("Same segment rows in both datasets:", same_rows)
print("Full rows:", len(full_keys))
print("Top-150 rows:", len(top150_keys))


Same segment rows in both datasets: True
Full rows: 406359
Top-150 rows: 406359


## Utility Baseline On The Same Data


In [7]:
utility_full = run_utility_baselines(
    features_df=full_features_df,
    target_col="utility_label",
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    models=FULL_FEATURE_MODELS,
)

utility_top150 = run_utility_baselines(
    features_df=top150_features_df,
    target_col="utility_label",
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    models=SUBSET_FEATURE_MODELS,
)

utility_full["summary_df"], utility_top150["summary_df"]


(                model  f1_score  f1_macro  balanced_accuracy   roc_auc  \
 0  LogisticRegression    0.6789  0.780105            0.83177  0.907671   
 
      pr_auc  
 0  0.712424  ,
                 model  f1_score  f1_macro  balanced_accuracy  roc_auc  \
 0  LogisticRegression  0.678106  0.779427           0.831456  0.90653   
 
      pr_auc  
 0  0.710244  )

## Linkability Baseline On The Same Data


In [8]:
linkability_full = run_linkability_baselines(
    features_df=full_features_df,
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
)

linkability_top150 = run_linkability_baselines(
    features_df=top150_features_df,
    group_col="patient_id",
    test_size=0.2,
    random_state=42,
    max_pairs=2000,
    representation="absdiff",
    min_segment_gap=4,
)

linkability_full["summary_df"], linkability_top150["summary_df"]


(                model  f1_score  f1_macro  balanced_accuracy   roc_auc  \
 0  LogisticRegression  0.975219  0.975250            0.97525  0.996190   
 1             XGBoost  0.978809  0.978998            0.97900  0.998343   
 
      pr_auc  
 0  0.996639  
 1  0.998426  ,
                 model  f1_score  f1_macro  balanced_accuracy   roc_auc  \
 0  LogisticRegression  0.969062  0.969000              0.969  0.993848   
 1             XGBoost  0.972864  0.972999              0.973  0.997051   
 
      pr_auc  
 0  0.994312  
 1  0.997157  )

## Compare Full Features vs Utility Top-150


In [9]:
utility_comparison = pd.concat([
    utility_full["summary_df"].assign(feature_set="full_208", task="utility"),
    utility_top150["summary_df"].assign(feature_set="utility_top150", task="utility"),
], ignore_index=True)

linkability_comparison = pd.concat([
    linkability_full["summary_df"].assign(feature_set="full_208", task="linkability"),
    linkability_top150["summary_df"].assign(feature_set="utility_top150", task="linkability"),
], ignore_index=True)

tradeoff_comparison_df = pd.concat([utility_comparison, linkability_comparison], ignore_index=True)
tradeoff_comparison_df


,model,f1_score,f1_macro,balanced_accuracy,roc_auc,pr_auc,feature_set,task
0,LogisticRegression,0.678900,0.780105,0.831770,0.907671,0.712424,full_208,utility
1,LogisticRegression,0.678106,0.779427,0.831456,0.906530,0.710244,utility_top150,utility
2,LogisticRegression,0.975219,0.975250,0.975250,0.996190,0.996639,full_208,linkability
3,XGBoost,0.978809,0.978998,0.979000,0.998343,0.998426,full_208,linkability
4,LogisticRegression,0.969062,0.969000,0.969000,0.993848,0.994312,utility_top150,linkability
5,XGBoost,0.972864,0.972999,0.973000,0.997051,0.997157,utility_top150,linkability


## Interpretation Notes

Use this table to answer:
- how much utility is lost when moving from the full feature set to the utility-focused subset?
- what happens to linkability when the same segment rows are represented with fewer features?

This keeps the comparison fair because the underlying data points are the same in both tasks.
